# Chapter 14 - Monitoring models (Tensorboard)

## What is TensorBoard?

**TensorBoard** is TensorFlow's visualization toolkit that provides a suite of web-based applications for inspecting and understanding your TensorFlow runs and graphs. It's an essential tool for:

- **Model Training Visualization**: Track metrics like loss and accuracy in real-time during training
- **Performance Profiling**: Identify computational bottlenecks and optimize model execution
- **Graph Visualization**: Understand the computational graph structure of your model
- **Data Inspection**: Visualize training data, including images, audio, and text
- **Hyperparameter Tuning**: Compare multiple training runs with different hyperparameters

## Why is Model Monitoring Important?

Monitoring models during training is a crucial part of the machine learning workflow. Continuous monitoring enables you to:

1. **Ensure Proper Training**: Verify that the model is learning correctly and not stuck in local minima
2. **Detect Overfitting Early**: Observe when validation metrics start diverging from training metrics
3. **Identify Performance Issues**: Spot computational bottlenecks that slow down training
4. **Make Data-Driven Decisions**: Use visual insights to improve model architecture and training strategies
5. **Debug Model Behavior**: Understand why a model might be failing or producing unexpected results

In this chapter, we will explore how to use TensorBoard to:
- Visualize various data types such as images and text
- Track training metrics and custom scalars
- Profile model performance to identify optimization opportunities
- Visualize high-dimensional embeddings like word vectors

# Important checks before running this code

## Setting the `TF_GPU_THREAD_MODE` variable

This variable will be something we'll be changing later in the code. The change you do will be persistent. Therefore, if you run this notebook multiple times, you'll be starting running the code with this variable set to a different value than the default. To avoid that, 
* Stop the Jupyter notebook server
* Set this environment variable `TF_GPU_THREAD_MODE=global` which is the default value. To do that, follow the instructions avalable at [this section](#set_environment) **with `global` as the value instead of `gpu_private`** to undo the changes.
* Restart the Juptyer notebook server

## Installing Model profiling with CUDA
In order to make sure all the features of the Tensorboard work, make sure to instell the `libcupti` library. It stands for **Lib**rary - **CU**DA **P**rofiling **T**ools **I**nterface. It is a GPU profiling toolkit by NVIDIA, which is required by the Tensboard profiling dashboard.

### Linux Installation - `libcupti`
On linux you can install this using `sudo apt-get install libcupti-dev`.

### Windows Installation - `libcupti`

As opposed to the Linux installation, Windows installation require more work.

* Make sure you have installed the required CUDA installation (e.g. CUDA 11 [>= TensorFlow 2.4.0])
* Next, open the NVIDIA Control Panel to do several changes (These were suggested in the following [Github issue](https://github.com/tensorflow/tensorflow/issues/35860#issuecomment-603728531)),
  * Make sure you have set the Developer Mode by clicking Desktop > Set Developer Mode
  * Make sure you have enabled GRU profiling to all users and not just the adiministrator. 
* For more errors you might face, refer the following page from the official [NVIDIA website](https://developer.nvidia.com/nvidia-development-tools-solutions-err-nvgpuctrperm-cupti)
* To install `libcupti` (Motivated by this [Stackoverflow question](https://stackoverflow.com/questions/54028188/how-to-install-cuda-profiling-tools-interface-on-windows-10/54029753)),
  * Copy `libcupti_<version>.dll`, `nvperf_host.dll` and `nvperf_target.dll` from the `extras\CUPTI\lib64` to the `bin` folder. Make sure the `libcupti` file has the name, `libcupti_110.dll`.
  * Copy all files in the `extras\CUPTI\lib64` to `lib\x64`
  * Copy all files in the `extras\CUPTI\include` to `include`.

# Importing necessary libraries

In [1]:
import random
import tensorflow as tf
import numpy as np
import tensorflow_datasets as tfds
import shutil
import os
from datetime import datetime

%load_ext tensorboard

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        # Currently, memory growth needs to be the same across GPUs
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except:
        print("Couldn't set memory_growth")
        pass
    
def fix_random_seed(seed):
    """ Setting the random seed of various libraries """
    try:
        np.random.seed(seed)
    except NameError:
        print("Warning: Numpy is not imported. Setting the seed for Numpy failed.")
    try:
        tf.random.set_seed(seed)
    except NameError:
        print("Warning: TensorFlow is not imported. Setting the seed for TensorFlow failed.")
    try:
        random.seed(seed)
    except NameError:
        print("Warning: random module is not imported. Setting the seed for random failed.")

# Fixing the random seed
random_seed=4321
fix_random_seed(random_seed)

log_datetimestamp_format = "%Y%m%d%H%M%S"
print("TensorFlow version: {}".format(tf.__version__))

TensorFlow version: 2.20.0


In [2]:
if os.path.exists('logs'):
    shutil.rmtree('logs')

# Visualizing Image Data on the TensorBoard

## Understanding Image Visualization in TensorBoard

TensorBoard's image dashboard allows you to visualize image data at various stages of your model pipeline. This is particularly useful for:

- **Inspecting Training Data**: Verify that your data preprocessing pipeline is working correctly
- **Debugging Data Augmentation**: See how augmentation transforms affect your images
- **Visualizing Model Outputs**: Compare input images with model predictions or reconstructions
- **Monitoring Feature Maps**: Visualize intermediate layer activations to understand what the model is learning

## How Image Logging Works

Images are logged to TensorBoard using the `tf.summary.image()` function, which writes image tensors to a specific directory. The TensorBoard server monitors this directory and displays the images in its web interface.

**Key concepts**:
- **Summary Writer**: Creates a file writer that logs data to a specific directory
- **Log Directory**: The location where TensorBoard data is stored
- **Step Parameter**: Allows you to track images across different training iterations or epochs
- **max_outputs**: Controls how many images from a batch are displayed

## Importing the Fashion-MNIST Dataset

Fashion-MNIST is a dataset of Zalando's article images, consisting of 70,000 grayscale images in 10 categories. It's commonly used as a drop-in replacement for MNIST and provides a more challenging classification task.

In [3]:
# Construct a tf.data.Dataset
fashion_ds = tfds.load('fashion_mnist')

print(fashion_ds)

d:\Source Code\ML\Tensor Flow in Action\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Dl Completed...: 0 url [00:00, ? url/s]
Dl Completed...:  25%|██▌       | 1/4 [00:01<00:03,  1.20s/ url]

Dl Completed...:  50%|█████     | 2/4 [00:02<00:02,  1.21s/ url]

Dl Completed...:  75%|███████▌  | 3/4 [00:14<00:05,  5.93s/ url]

Dl Completed...: 100%|██████████| 4/4 [03:08<00:00, 72.46s/ url]

Dl Completed...: 100%|██████████| 4/4 [03:08<00:00, 47.18s/ url]


Dataset fashion_mnist downloaded and prepared to C:\Users\nakir\tensorflow_datasets\fashion_mnist\3.0.1. Subsequent calls will reuse this data.
{Split('train'): <_PrefetchDataset element_spec={'image': TensorSpec(shape=(28, 28, 1), dtype=tf.uint8, name=None), 'label': TensorSpec(shape=(), dtype=tf.int64, name=None)}>, Split('test'): <_PrefetchDataset element_spec={'image': TensorSpec(shape=(28, 28, 1), dtype=tf.uint8, name=None), 'label': TensorSpec(shape=(), dtype=tf.int64, name=None)}>}


## Create training/validation/testing data

As we have done before, let's separate the data to training, validation and testing subsets.

In [4]:
# Section 14.1

# Code listing 14.1
def get_train_valid_test_datasets(fashion_ds, batch_size, flatten_images=False):
    
    # Get the training dataset, shuffle it, and output a tuple of (image, label) 
    train_ds = fashion_ds["train"].shuffle(batch_size*20).map(lambda xy: (xy["image"], tf.reshape(xy["label"], [-1])))
    # Get the testing dataset, and output a tuple of (image, label)
    test_ds = fashion_ds["test"].map(lambda xy: (xy["image"], tf.reshape(xy["label"], [-1])))
    
    if flatten_images:
        # Flatten the images to a 1D vector for fully-connected networks
        train_ds = train_ds.map(lambda x,y: (tf.reshape(x, [-1]), y))
        test_ds = test_ds.map(lambda x,y: (tf.reshape(x, [-1]), y))
    
    # Make the validation dataset the first 10000 data
    valid_ds = train_ds.take(10000).batch(batch_size)
    # Make training dataset the rest
    train_ds = train_ds.skip(10000).batch(batch_size).prefetch(tf.data.experimental.AUTOTUNE)
    
    return train_ds, valid_ds, test_ds

## Using `tf.summary` to visualize images on TensorBoard

### The tf.summary API

The `tf.summary` module provides the core functionality for logging data to TensorBoard. It supports various data types:
- **Scalars**: Single numerical values (loss, accuracy, learning rate)
- **Images**: Image tensors for visual inspection
- **Histograms**: Distributions of weights and activations
- **Text**: String data for debugging or displaying model outputs
- **Audio**: Audio waveforms
- **Embeddings**: High-dimensional vectors for visualization

### Image Logging Workflow

1. **Create a File Writer**: Use `tf.summary.create_file_writer(logdir)` to specify where logs should be written
2. **Open the Writer Context**: Use `with writer.as_default():` to activate the writer
3. **Log Images**: Call `tf.summary.image(name, data, step, max_outputs)` to log image tensors
4. **Flush to Disk**: The writer automatically flushes data, but you can manually call `writer.flush()`

**Important Parameters**:
- `name`: A descriptive name for the image set (appears as a tag in TensorBoard)
- `data`: The image tensor (must have shape [batch, height, width, channels])
- `step`: The training step or iteration number (allows tracking changes over time)
- `max_outputs`: Maximum number of images to display from the batch

In [5]:
# Section 14.1

# Defining the ID to Label map
id2label_map = {
    0: "T-shirt/top",
    1: "Trouser",
    2:"Pullover",
    3: "Dress",
    4: "Coat",
    5: "Sandal",
    6: "Shirt",
    7: "Sneaker",
    8: "Bag",
    9: "Ankle boot"
}

print("Writing to the tensorboard")

log_datetimestamp = datetime.strftime(datetime.now(), log_datetimestamp_format)
image_logdir = "./logs/data_{}/train".format(log_datetimestamp)

# Define a summary writer
image_writer = tf.summary.create_file_writer(image_logdir)

# Write an image with its category
with image_writer.as_default():
    for data in fashion_ds["train"].batch(1).take(10):
        tf.summary.image(id2label_map[int(data["label"].numpy())], data["image"], max_outputs=20, step=0)

# Write a batch of images at once
with image_writer.as_default():
    for data in fashion_ds["train"].batch(20).take(1):
        pass
    tf.summary.image("A training data batch", data["image"], max_outputs=20, step=0)

print('\tDone')

Writing to the tensorboard


C:\Users\nakir\AppData\Local\Temp\ipykernel_23188\1307804291.py:28: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  tf.summary.image(id2label_map[int(data["label"].numpy())], data["image"], max_outputs=20, step=0)


	Done


# Spinning up the TensorBoard
 
Here we're using tensorboard magic command on jupyter notebook. This gives us the TensorBoard inline, as if you were to open the Tensorboard in a browser tab. If you call the same command multiple times with the same `logdir` it will reuse the same Tensorboard. If the directories are different a new TensorBoard is spun up. 

There are times you have to restart the TensorBoard to get a fresh view of the logged data. For that,

On Linux,
* Open a command line terminal and execute `ps -ef|grep tensorboard`. This will give the process ID of TensorBoard
* Execute `kill -9 <TensorBoard process ID>` to kill the process.

On Windows,
* Execute the following two lines in the Jupyter notebook
* `!taskkill /IM "tensorboard.exe" /F`
* `!rmdir /s /q C:\Users\<user name>\AppData\Local\Temp\.tensorboard-info`

**Note**: On windows, it's not just enough to kill the process to restart the tensorboard. You have to delete the `C:\Users\<user name>\AppData\Local\Temp\.tensorboard-info` directory as well.

In [6]:
%tensorboard --logdir ./logs --port 6006

# Tracking models on TensorBoard

## Understanding Model Comparison

One of TensorBoard's most powerful features is the ability to compare multiple models or training runs side-by-side. This enables you to:

- **Compare Architectures**: See how different model designs perform on the same task
- **Evaluate Hyperparameters**: Test various learning rates, batch sizes, or optimizer configurations
- **Identify Best Practices**: Discover which techniques (dropout, batch normalization, etc.) improve performance
- **Make Informed Decisions**: Use empirical evidence to guide model development

### The TensorBoard Callback

The `tf.keras.callbacks.TensorBoard` callback automatically logs metrics during training without requiring manual summary writing. Key features:

**Automatic Logging**:
- Training and validation loss
- All metrics specified in `model.compile()`
- Learning rate schedules

**Optional Features**:
- `histogram_freq`: Log weight/bias distributions every N epochs
- `write_graph`: Visualize the model architecture
- `profile_batch`: Profile computational performance for specific batches
- `embeddings_freq`: Log embeddings for visualization

## Model Comparison: Fully-Connected vs CNN

Here we will compare two models: 
* A **fully-connected (dense) network**: Simple but may not capture spatial features effectively
* A **convolutional neural network (CNN)**: Designed to exploit spatial structure in images

By comparing these architectures on Fashion-MNIST, we can observe:
- How CNNs converge faster due to parameter sharing and local connectivity
- The difference in parameter count and computational cost
- How architecture choice affects final accuracy

## Monitoring the performance of the fully-connected network

### Fully-connected network architecture

This network consists of three fully-connected layers with:
- **Layer 1**: 512 neurons with ReLU activation
- **Layer 2**: 256 neurons with ReLU activation  
- **Layer 3**: 10 neurons with softmax (for 10 classes)

The model uses flattened 28×28 images as input (784 features).

In [7]:
# Section 14.2

from tensorflow.keras import layers, models


dense_model = models.Sequential([
    layers.Dense(512, activation='relu', input_shape=(784,)),
    layers.Dense(256, activation='relu'),
    layers.Dense(10, activation='softmax')
])

dense_model.compile(loss="sparse_categorical_crossentropy", optimizer='adam', metrics=['accuracy'])


d:\Source Code\ML\Tensor Flow in Action\.venv\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


## Training the model

In [8]:
# Section 14.2

log_datetimestamp = datetime.strftime(datetime.now(), log_datetimestamp_format)
dense_log_dir = os.path.join("logs","dense_{}".format(log_datetimestamp))

batch_size = 64
tr_ds, v_ds, ts_ds = get_train_valid_test_datasets(fashion_ds, batch_size=batch_size, flatten_images=True)

# Defining the tensorboard callback, it will log information to the defined log_dir directory
tb_callback = tf.keras.callbacks.TensorBoard(log_dir=dense_log_dir, profile_batch=0)

# Train the model
dense_model.fit(tr_ds, validation_data=v_ds, epochs=10, callbacks=[tb_callback])


Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.7664 - loss: 3.7409 - val_accuracy: 0.8176 - val_loss: 0.6565
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8180 - loss: 0.5887 - val_accuracy: 0.8150 - val_loss: 0.5659
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8330 - loss: 0.4982 - val_accuracy: 0.8200 - val_loss: 0.5765
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8383 - loss: 0.4744 - val_accuracy: 0.8224 - val_loss: 0.5354
Epoch 5/10
134/782 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8508 - loss: 0.4443

KeyboardInterrupt: 

To view the results of the fully connected model,

---
## Open [Tensorboard](http://localhost:6006) in the browser
---

## Monitoring the performance of the CNN

### Why CNNs Outperform Fully-Connected Networks on Images

Convolutional Neural Networks have several advantages for image data:

1. **Parameter Efficiency**: CNNs use shared weights across spatial locations, requiring far fewer parameters than fully-connected networks
2. **Spatial Invariance**: Convolutional filters detect features regardless of their position in the image
3. **Hierarchical Features**: Early layers learn simple patterns (edges, textures) while deeper layers combine them into complex features
4. **Locality**: Convolutions exploit the fact that nearby pixels are more related than distant ones

### CNN Architecture

This CNN uses:
- **Conv2D Layer 1**: 32 filters, 5×5 kernel, stride 2, ReLU activation
- **Conv2D Layer 2**: 16 filters, 3×3 kernel, stride 1, ReLU activation
- **Flatten**: Convert 2D feature maps to 1D vector
- **Dense Output**: 10 neurons with softmax activation

### Histogram Visualization

By setting `histogram_freq=2`, we tell TensorBoard to log weight and bias distributions every 2 epochs. This helps us:
- **Detect Vanishing/Exploding Gradients**: See if weights are growing or shrinking uncontrollably
- **Monitor Layer Activity**: Identify layers that aren't learning (flat histograms)
- **Verify Proper Initialization**: Ensure weights start with appropriate distributions

In [ ]:
# Section 14.2

import tensorflow.keras.backend as K
K.clear_session()

conv_model = models.Sequential([
    layers.Conv2D(filters=32, kernel_size=(5,5), strides=(2,2), padding='same', activation='relu', input_shape=(28,28,1)),
    layers.Conv2D(filters=16, kernel_size=(3,3), strides=(1,1), padding='same', activation='relu'),
    layers.Flatten(),
    layers.Dense(10, activation='softmax')
])

conv_model.compile(loss="sparse_categorical_crossentropy", optimizer='adam', metrics=['accuracy'])
conv_model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d (Conv2D)              (None, 14, 14, 32)        832       
_________________________________________________________________
conv2d_1 (Conv2D)            (None, 14, 14, 16)        4624      
_________________________________________________________________
flatten (Flatten)            (None, 3136)              0         
_________________________________________________________________
dense (Dense)                (None, 10)                31370     
Total params: 36,826
Trainable params: 36,826
Non-trainable params: 0
_________________________________________________________________


## Training the model

In [ ]:
log_datetimestamp = datetime.strftime(datetime.now(), log_datetimestamp_format)
conv_log_dir = os.path.join("logs","conv_{}".format(log_datetimestamp))

In [ ]:
batch_size = 64
tr_ds, v_ds, ts_ds = get_train_valid_test_datasets(fashion_ds, batch_size=batch_size, flatten_images=False)

# This tensorboard call back does the followin
# 1. Log loss and accuracy
# 2. Plot activation histograms every two epochs
tb_callback = tf.keras.callbacks.TensorBoard(log_dir=conv_log_dir, histogram_freq=2, profile_batch=0)

conv_model.fit(tr_ds, validation_data=v_ds, epochs=10, callbacks=[tb_callback])

Epoch 1/10
782/782 [==============================] - 6s 7ms/step - loss: 0.6403 - accuracy: 0.8196 - val_loss: 0.4051 - val_accuracy: 0.8575
Epoch 2/10
782/782 [==============================] - 6s 7ms/step - loss: 0.3471 - accuracy: 0.8773 - val_loss: 0.3593 - val_accuracy: 0.8759
Epoch 3/10
782/782 [==============================] - 6s 7ms/step - loss: 0.3018 - accuracy: 0.8918 - val_loss: 0.3591 - val_accuracy: 0.8808
Epoch 4/10
782/782 [==============================] - 6s 7ms/step - loss: 0.2787 - accuracy: 0.8992 - val_loss: 0.3735 - val_accuracy: 0.8761
Epoch 5/10
782/782 [==============================] - 6s 7ms/step - loss: 0.2608 - accuracy: 0.9059 - val_loss: 0.3999 - val_accuracy: 0.8741
Epoch 6/10
782/782 [==============================] - 6s 7ms/step - loss: 0.2434 - accuracy: 0.9107 - val_loss: 0.3707 - val_accuracy: 0.8836
Epoch 7/10
782/782 [==============================] - 6s 7ms/step - loss: 0.2372 - accuracy: 0.9125 - val_loss: 0.3631 - val_accuracy: 0.8848
Epoch 

To view the result comparison between the fully connected model and the CNN,

---
## Open [Tensorboard](http://localhost:6006) in the browser
---

# Logging custom metrics to the TensorBoard

## Understanding Custom Metrics

While TensorBoard's automatic logging is convenient, sometimes you need to track custom metrics that aren't automatically computed. Common use cases include:

- **Weight Statistics**: Mean, standard deviation, or norms of weight matrices
- **Gradient Information**: Gradient magnitudes to detect vanishing/exploding gradients
- **Custom Loss Components**: Individual terms in a complex loss function
- **Data Statistics**: Properties of your training batches
- **Performance Metrics**: Custom evaluation criteria specific to your problem

## Batch Normalization Analysis

**Batch Normalization** is a technique that normalizes layer inputs during training, which:
- Reduces internal covariate shift (changing input distributions)
- Allows higher learning rates
- Acts as a form of regularization
- Stabilizes training and often improves convergence

### Experiment Design

We'll train two identical networks:
1. **Standard Model**: No batch normalization
2. **Batch Normalized Model**: Batch normalization after each dense layer

### What We're Measuring

By tracking the **mean and standard deviation of absolute weights** in the second layer, we can observe:
- **Weight Stability**: How batch normalization affects weight magnitudes over time
- **Learning Dynamics**: Whether batch normalization leads to more stable weight updates
- **Convergence Behavior**: If batch normalization helps weights settle faster

### Custom Logging with tf.summary.scalar

The `tf.summary.scalar()` function logs single numerical values. Key parameters:
- `name`: The metric name (appears in TensorBoard)
- `data`: The numerical value to log
- `step`: The training iteration or batch number

**Important**: Custom logging requires:
1. Creating a file writer with `tf.summary.create_file_writer()`
2. Opening the writer context with `with writer.as_default():`
3. Calling `tf.summary.scalar()` within that context
4. Optionally calling `writer.flush()` to ensure data is written to disk

In [ ]:
# Section 14.3

from tensorflow.keras import layers, models
import tensorflow.keras.backend as K

K.clear_session()

dense_model = models.Sequential([
    layers.Dense(512, activation='relu', input_shape=(784,)),    
    layers.Dense(256, activation='relu', name='log_layer'),    
    layers.Dense(10, activation='softmax')
])

dense_model.compile(loss="sparse_categorical_crossentropy", optimizer='adam', metrics=['accuracy'])

dense_model_bn = models.Sequential([
    layers.Dense(512, activation='relu', input_shape=(784,)),
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu', name='log_layer_bn'),
    layers.BatchNormalization(),
    layers.Dense(10, activation='softmax')
])

dense_model_bn.compile(loss="sparse_categorical_crossentropy", optimizer='adam', metrics=['accuracy'])

## Training the model

In [ ]:
log_datetimestamp = datetime.strftime(datetime.now(), log_datetimestamp_format)
exp_log_dir = os.path.join("logs","weights_exp_{}".format(log_datetimestamp))

In [ ]:
# Section 14.3

# Code listing 14.2
def train_model(model, dataset, log_dir, log_layer_name, epochs):    
    
    # Define the writer
    writer = tf.summary.create_file_writer(log_dir)
    
    step = 0
    # Open the writer
    with writer.as_default():        
        tot_iterations_in_epoch = 0  # Total iterations in an epoch
        
        # For every epoch
        for e in range(epochs):
            print("Training epoch {}".format(e+1))
            # For every iteration in the epoch
            for batch in tr_ds:
                # Compute the step
                
                # Train with one batch
                model.train_on_batch(*batch)
                # Get the weights of the layer [0] - weights / [1] - bias
                w = model.get_layer(log_layer_name).get_weights()[0]
                
                # Log mean and std of absolute weights
                tf.summary.scalar("mean_weights", np.mean(np.abs(w)), step=step)
                tf.summary.scalar("std_weights", np.std(np.abs(w)), step=step)
                
                # Flush to the disk from the buffer
                writer.flush()
                
                step += 1
            print('\tDone')
    
    print("Training completed\n")
    
batch_size = 64
tr_ds, _, _ = get_train_valid_test_datasets(fashion_ds, batch_size=batch_size, flatten_images=True)
train_model(dense_model, tr_ds, exp_log_dir + '/standard', "log_layer", 5)

tr_ds, _, _ = get_train_valid_test_datasets(fashion_ds, batch_size=batch_size, flatten_images=True)
train_model(dense_model_bn, tr_ds, exp_log_dir + '/bn', "log_layer_bn", 5)

Training epoch 1
	Done
Training epoch 2
	Done
Training epoch 3
	Done
Training epoch 4
	Done
Training epoch 5
	Done
Training completed

Training epoch 1
	Done
Training epoch 2
	Done
Training epoch 3
	Done
Training epoch 4
	Done
Training epoch 5
	Done
Training completed



Don't forget that you can look at the results in the [TensorBoard](http://localhost:6006)


# Profiling models to detect performance bottlenecks

## Understanding Model Profiling

**Profiling** is the process of measuring where your model spends time and resources during training. TensorBoard's profiler helps you:

### Key Benefits of Profiling

1. **Identify Bottlenecks**: Find the slowest operations in your pipeline
2. **Optimize Resource Usage**: Ensure GPU utilization is high
3. **Reduce Training Time**: Discover opportunities for optimization
4. **Debug Performance Issues**: Understand why training is slower than expected

### What the Profiler Measures

- **Step Time**: Total time per training step
- **GPU Utilization**: Percentage of time GPU is actively computing
- **Memory Usage**: RAM and GPU memory consumption
- **Op Performance**: Time spent in individual TensorFlow operations
- **Input Pipeline**: Time spent loading and preprocessing data
- **Kernel Launch**: GPU kernel execution time

### Common Performance Issues

1. **Input Pipeline Bottleneck**: CPU can't feed data fast enough to GPU
2. **Small Batch Sizes**: GPU underutilized due to insufficient parallelism
3. **Host-Device Transfers**: Too much data movement between CPU and GPU
4. **Inefficient Operations**: Using slow ops when faster alternatives exist
5. **Lack of Mixed Precision**: Not leveraging Tensor Cores on modern GPUs

## The Flowers Dataset

For this profiling exercise, we'll use a more complex CNN on the 17-category flowers dataset. This larger model will make profiling insights more apparent.

In [ ]:
# Section 14.4

# Downloading the data

import os
import requests
import tarfile

import shutil

# Retrieve the data
if not os.path.exists(os.path.join('data', '17flowers.tgz')):
    
    url="https://www.robots.ox.ac.uk/~vgg/data/flowers/17/17flowers.tgz"

    # Get the file from web
    r = requests.get(url)

    if not os.path.exists('data'):
        os.makedirs('data')

    # Write to a file
    with open(os.path.join('data', '17flowers.tgz'), 'wb') as f:
        f.write(r.content)

else:
    print("The tar file already exists.")

if not os.path.exists(os.path.join('data', '17flowers')):
    # Write to a file
    tarf = tarfile.open(os.path.join("data","17flowers.tgz"))
    tarf.extractall(os.path.join('data', '17flowers'))
else:
    print("The extracted data already exists")

The tar file already exists.
The extracted data already exists


## Define the CNN model

In [ ]:
# Code listing 14.3
def get_cnn_model():
    
    conv_model = models.Sequential([
        layers.Conv2D(filters=64, kernel_size=(5,5), strides=(1,1), padding='same', activation='relu', input_shape=(64,64,3)),
        layers.BatchNormalization(),
        layers.MaxPooling2D(pool_size=(3,3), strides=(2,2)),
        layers.Conv2D(filters=128, kernel_size=(3,3), strides=(1,1), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(filters=256, kernel_size=(3,3), strides=(1,1), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(filters=512, kernel_size=(3,3), strides=(1,1), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.AveragePooling2D(pool_size=(2,2), strides=(2,2)),
        layers.Flatten(),        
        layers.Dense(512),
        layers.LeakyReLU(),
        layers.LayerNormalization(),                
        layers.Dense(256),
        layers.LeakyReLU(),
        layers.LayerNormalization(),                
        layers.Dense(17),
        layers.Activation('softmax', dtype='float32')
    ])
    return conv_model

In [ ]:
print(os.environ["TF_GPU_THREAD_MODE"])

global


In [ ]:
# Section 14.4

import os
from tensorflow.keras import layers, models
import tensorflow.keras.backend as K
K.clear_session()

profile_log_dir = os.path.join("logs","profile")
    
conv_model = get_cnn_model()

conv_model.compile(loss="sparse_categorical_crossentropy", optimizer='adam', metrics=['accuracy'])
#conv_model.summary()

def get_flower_datasets(image_dir, batch_size, flatten_images=False):

    # Get the training dataset, shuffle it, and output a tuple of (image, label)
    dataset = tf.data.Dataset.list_files(os.path.join(image_dir,'*.jpg'), shuffle=False)

    def get_image_and_label(file_path):

        tokens = tf.strings.split(file_path, os.path.sep)        
        label = (tf.strings.to_number(tf.strings.split(tf.strings.split(tokens[-1],'.')[0], '_')[-1])-1)//80

        # load the raw data from the file as a string
        img = tf.io.read_file(file_path)
        img = tf.image.decode_jpeg(img, channels=3)

        return tf.image.resize(img, [64, 64]), label

    dataset = dataset.map(get_image_and_label).shuffle(400)

    # Make the validation dataset the first 10000 data
    valid_ds = dataset.take(250).batch(batch_size)
    # Make training dataset the rest
    train_ds = dataset.skip(250).batch(batch_size)

    return train_ds, valid_ds

batch_size = 32
tr_ds, v_ds = get_flower_datasets(
    os.path.join('data', '17flowers','jpg'), batch_size=batch_size, flatten_images=False
)
    
# This tensorboard call back does the followin
# 1. Log loss and accuracy
# 2. Profile the model memory/time for 10 batches
tb_callback = tf.keras.callbacks.TensorBoard(log_dir=profile_log_dir, profile_batch=[10, 20])

conv_model.fit(tr_ds, validation_data=v_ds, epochs=2, callbacks=[tb_callback])


Epoch 1/2
35/35 [==============================] - 38s 1s/step - loss: 2.9206 - accuracy: 0.3135 - val_loss: 3.1460 - val_accuracy: 0.0280
Epoch 2/2
35/35 [==============================] - 38s 1s/step - loss: 1.9648 - accuracy: 0.3775 - val_loss: 2.7163 - val_accuracy: 0.1640


## Improving the CNN backed up by TensorBoard profiler findings

### Performance Optimization Strategies

Based on TensorBoard profiler insights, we'll implement three key optimizations:

#### 1. Optimize the tf.data Pipeline

**Problem**: The input pipeline may become a bottleneck if the CPU can't prepare data fast enough for the GPU.

**Solutions**:
- **Prefetching**: Use `.prefetch()` to prepare the next batch while the current batch is being processed
- **Parallel Map**: Use `num_parallel_calls=AUTOTUNE` to parallelize data preprocessing
- **Caching**: Cache preprocessed data in memory to avoid redundant computation

#### 2. Use Mixed Precision Training

**What is Mixed Precision?**

Mixed precision training uses both 16-bit (float16) and 32-bit (float32) floating-point types:
- **float16** for most computations → Faster and uses less memory
- **float32** for numerical stability in critical operations

**Benefits**:
- **2-3x Speedup**: On GPUs with Tensor Cores (compute capability ≥ 7.0)
- **Reduced Memory Usage**: Allows larger batch sizes or models
- **Maintained Accuracy**: Automatic loss scaling prevents underflow

**Requirements**:
- GPU with compute capability ≥ 7.0 (e.g., V100, RTX 2070+, A100)
- TensorFlow 2.4+ with proper CUDA installation

**GPU Compatibility Check**:

GPUs with compute capability < 7.0 will show a warning:
```
WARNING:tensorflow:Mixed precision compatibility check (mixed_float16): WARNING
Your GPU may run slowly with dtype policy mixed_float16 because it does not have compute capability of at least 7.0.
```

GPUs with compute capability ≥ 7.0 will show:
```
INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0.
```

#### 3. Use GPU Private Thread Mode

**What is TF_GPU_THREAD_MODE?**

This environment variable controls how TensorFlow launches GPU kernels:
- **global** (default): Uses a single thread pool shared across all GPUs
- **gpu_private**: Each GPU gets its own thread pool

**When to use gpu_private**:
- Training with multiple GPUs
- Reduces kernel launch latency
- Better for models with many small operations

**Trade-off**: Slightly higher CPU overhead, but often worth it for complex models.

<a id="set_environment"></a>

## Setting Environment Variables

To set environment variables you can do the following.

### Linux

Set the environment variable by,
* Opening a terminal 
* Run `export TF_GPU_THREAD_MODE=gpu_private`
* Verify the environment variable is set by calling `echo $TF_GPU_THREAD_MODE`
* Open a new shell and start the jupyter notebook server

### Windows

Set the environment variable by,
* From the start menu select `Edit the system environment variables`
* Click the button called `environment variables`
* Add a new environment variable `TF_GPU_THREAD_MODE=gpu_private` in the opened dialog
* Open a new command prompt and start the jupyter notebook server

### Conda environment

To set environment variables in a conda environment,
* Activate the conda environment with `conda activate manning.tf2`
* Run `conda env config vars set TF_GPU_THREAD_MODE=gpu_private`
* Deactivate and reactivate the environment, for the variable to take effect
* Start the jupyter notebook server

In [ ]:
opt_profile_log_dir = os.path.join("logs","optimized_profile")

In [ ]:
# Section 14.4

from tensorflow.keras import mixed_precision
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)

from tensorflow.keras import layers, models
import tensorflow.keras.backend as K
K.clear_session()

conv_model = get_cnn_model()

conv_model.compile(loss="sparse_categorical_crossentropy", optimizer='adam', metrics=['accuracy'])

# Code listing 14.4
def get_flower_datasets(image_dir, batch_size, flatten_images=False):

    # Get the training dataset, shuffle it, and output a tuple of (image, label)
    dataset = tf.data.Dataset.list_files(os.path.join(image_dir,'*.jpg'), shuffle=False)

    def get_image_and_label(file_path):

        tokens = tf.strings.split(file_path, os.path.sep)        
        label = (tf.strings.to_number(tf.strings.split(tf.strings.split(tokens[-1],'.')[0], '_')[-1])-1)//80

        # load the raw data from the file as a string
        img = tf.io.read_file(file_path)
        img = tf.image.decode_jpeg(img, channels=3)

        return tf.image.resize(img, [64, 64]), label

    dataset = dataset.map(
        get_image_and_label,
        num_parallel_calls=tf.data.AUTOTUNE
    ).shuffle(400)

    # Make the validation dataset the first 10000 data
    valid_ds = dataset.take(250).batch(batch_size)
    # Make training dataset the rest
    train_ds = dataset.skip(250).batch(batch_size).prefetch(tf.data.experimental.AUTOTUNE)

    return train_ds, valid_ds

batch_size = 32
tr_ds, v_ds = get_flower_datasets(os.path.join('data', '17flowers','jpg'), batch_size=batch_size, flatten_images=False)

# This tensorboard call back does the followin
# 1. Log loss and accuracy
# 2. Profile the model memory/time for 370-410 batches
tb_callback = tf.keras.callbacks.TensorBoard(log_dir=opt_profile_log_dir, profile_batch=[10, 20])

conv_model.fit(tr_ds, validation_data=v_ds, epochs=2, callbacks=[tb_callback])

# Resetting to float32
policy = mixed_precision.Policy('float32')
mixed_precision.set_global_policy(policy)


INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: GeForce RTX 2070, compute capability 7.5


INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: GeForce RTX 2070, compute capability 7.5


Epoch 1/2
35/35 [==============================] - 12s 167ms/step - loss: 3.5535 - accuracy: 0.2541 - val_loss: 3.8611 - val_accuracy: 0.0000e+00
Epoch 2/2
35/35 [==============================] - 4s 88ms/step - loss: 2.1774 - accuracy: 0.3194 - val_loss: 3.3964 - val_accuracy: 0.0000e+00


## Checking the data types when using mixed precision training

### Understanding Automatic Mixed Precision

When you enable mixed precision with `mixed_float16` policy, TensorFlow automatically manages data types:

#### Data Type Flow in Mixed Precision

1. **Input Tensors**: Automatically cast to **float16** for faster computation
2. **Layer Variables (Weights/Biases)**: Stored as **float32** for numerical stability
3. **Computations**: Performed in **float16** to leverage Tensor Cores
4. **Output Tensors**: Usually **float16**, but final layers may output **float32**

#### Why This Matters

- **float16 computations** are 2-3x faster on modern GPUs
- **float32 variables** prevent weight updates from becoming too small (underflow)
- TensorFlow handles conversions automatically—you don't need to modify your code!

#### Loss Scaling

Mixed precision training also uses **loss scaling** (handled automatically):
- Multiplies loss by a large factor before backpropagation
- Prevents gradients from becoming too small and underflowing
- Divides gradients by the same factor before applying updates

This ensures training stability without sacrificing the speed benefits of float16.

In [ ]:
# Section 14.4

print("Input to the layers have the data type: {}".format(conv_model.get_layer("conv2d_1").input.dtype))
print("Variables in the layers have the data type: {}".format(conv_model.get_layer("conv2d_1").trainable_variables[0].dtype))
print("Output of the layers have the data type: {}".format(conv_model.get_layer("conv2d_1").output.dtype))

Input to the layers have the data type: <dtype: 'float16'>
Variables in the layers have the data type: <dtype: 'float32'>
Output of the layers have the data type: <dtype: 'float16'>


# Visualizing word vectors on TensorBoard

## Understanding Word Embeddings

**Word embeddings** are dense vector representations of words where similar words have similar vectors. They capture semantic relationships:
- Words with similar meanings are close in vector space
- Analogies like "king - man + woman ≈ queen" emerge naturally
- Dimensions encode latent concepts (gender, plurality, tense, etc.)

### Why Visualize Embeddings?

Visualization helps you:
1. **Understand Model Behavior**: See what semantic relationships your model has learned
2. **Debug Issues**: Identify if certain words are mapped incorrectly
3. **Explore Patterns**: Discover unexpected clusters and relationships
4. **Validate Training**: Ensure embeddings make linguistic sense

## TensorBoard's Embedding Projector

TensorBoard provides a dedicated **Projector** tool for visualizing high-dimensional data:

### Dimensionality Reduction Methods

Since we can't directly visualize 50+ dimensions, the projector uses:

1. **PCA (Principal Component Analysis)**:
   - Linear dimensionality reduction
   - Preserves global structure
   - Fast computation
   - Good for initial exploration

2. **t-SNE (t-Distributed Stochastic Neighbor Embedding)**:
   - Non-linear dimensionality reduction
   - Excellent for visualizing local clusters
   - Preserves neighborhood relationships
   - Computationally expensive but highly effective

3. **UMAP (Uniform Manifold Approximation and Projection)**:
   - Modern alternative to t-SNE
   - Faster than t-SNE
   - Better preserves global structure
   - Increasingly popular for embedding visualization

### Interactive Features

The projector allows you to:
- **Search**: Find specific words in the embedding space
- **Nearest Neighbors**: Click a word to see its closest semantic neighbors
- **3D Rotation**: Explore embeddings from different angles
- **Metadata**: Color-code points by categories or properties

## Download GloVe word vectors

### What are GloVe Vectors?

**GloVe** (Global Vectors for Word Representation) is an unsupervised learning algorithm for obtaining word embeddings. Key characteristics:

- **Developed by**: Stanford NLP Group (Pennington, Socher, Manning, 2014)
- **Training Method**: Matrix factorization on word co-occurrence statistics
- **Philosophy**: "You shall know a word by the company it keeps"
- **Pre-trained**: Available for billions of tokens from web crawls

### GloVe vs Word2Vec

| Feature | GloVe | Word2Vec |
|---------|-------|----------|
| Method | Matrix factorization | Neural network |
| Input | Global co-occurrence matrix | Local context windows |
| Training | Faster | Slower |
| Performance | Similar quality | Similar quality |

### Available GloVe Models

The `glove.6B.zip` file contains multiple embedding sizes:
- **glove.6B.50d.txt**: 50-dimensional vectors (we'll use this)
- **glove.6B.100d.txt**: 100-dimensional vectors
- **glove.6B.200d.txt**: 200-dimensional vectors
- **glove.6B.300d.txt**: 300-dimensional vectors

Higher dimensions capture more nuance but are computationally expensive.

In [ ]:
# Section 14.5

import os
import requests
import zipfile

if not os.path.exists(os.path.join('data','glove.6B.zip')):
    
    print("Downloading")
    url = "http://nlp.stanford.edu/data/glove.6B.zip"
    # Get the file from web
    r = requests.get(url)

    if not os.path.exists('data'):
        os.mkdir('data')
    
    # Write to a file
    with open(os.path.join('data','glove.6B.zip'), 'wb') as f:
        f.write(r.content)
    print("\tDone")
    
else:
    print("The zip file already exists.")
    
if not os.path.exists(os.path.join('data', 'glove.6B.50d.txt')):
    print("Extracting data")
    with zipfile.ZipFile(os.path.join('data','glove.6B.zip'), 'r') as zip_ref:
        zip_ref.extractall('data')
    print("\tDone")
else:
    print("The extracted data already exists")

The zip file already exists.
The extracted data already exists


## Getting the most common words in the IMDB movie review dataset

### Why Use IMDB Reviews?

The **IMDB movie review dataset** contains 50,000 reviews with sentiment labels (positive/negative). By analyzing word vectors from this domain:
- We can see how sentiment-bearing words cluster together
- Observe relationships between movie-related terminology
- Understand which words are most important for this domain

### Focusing on Common Words

We extract the **most common 5000 words** because:
1. **Visualization Clarity**: Too many points make the projection cluttered
2. **Meaningful Context**: Common words are more likely to have well-defined relationships
3. **Computational Efficiency**: Reduces processing time for dimensionality reduction
4. **Domain Relevance**: These words capture the core vocabulary of movie reviews

### Word Frequency Analysis

Using Python's `Counter` class, we:
- Tokenize all reviews into individual words
- Count occurrences of each unique word
- Select the top 5000 by frequency
- These become our vocabulary for embedding visualization

In [ ]:
import numpy as np
import pandas as pd

review_ds = tfds.load('imdb_reviews')
train_review_ds = review_ds["train"]

corpus = []
for data in train_review_ds:      
    txt = str(np.char.decode(data["text"].numpy(), encoding='utf-8')).lower()
    corpus.append(str(txt))

We will use the most common 5000 words as our sample

In [ ]:
from collections import Counter

corpus = " ".join(corpus)

cnt = Counter(corpus.split())
print(cnt.most_common(100))

most_common_words = [w for w,_ in cnt.most_common(5000)]

[('the', 322198), ('a', 159953), ('and', 158572), ('of', 144462), ('to', 133967), ('is', 104171), ('in', 90527), ('i', 70480), ('this', 69714), ('that', 66292), ('it', 65505), ('/><br', 50935), ('was', 47024), ('as', 45102), ('for', 42843), ('with', 42729), ('but', 39764), ('on', 31619), ('movie', 30887), ('his', 29059), ('are', 28743), ('not', 28597), ('film', 27777), ('you', 27564), ('have', 27344), ('he', 26177), ('be', 25691), ('at', 22731), ('one', 22480), ('by', 21976), ('an', 21240), ('they', 20624), ('from', 19934), ('all', 19740), ('who', 19407), ('like', 18779), ('so', 18099), ('just', 17309), ('or', 16769), ('has', 16570), ('her', 16540), ('about', 16486), ("it's", 15970), ('some', 15280), ('if', 15189), ('out', 14510), ('what', 14055), ('very', 13633), ('when', 13609), ('more', 13170), ('there', 13094), ('she', 12234), ('would', 12027), ('even', 12010), ('good', 11926), ('my', 11766), ('only', 11566), ('their', 11317), ('no', 11273), ('really', 11065), ('had', 11042), ('whi

## Read GloVe vectors 

### Filtering to Relevant Vocabulary

Here we perform vocabulary intersection:
1. **Read GloVe**: Load all 400,000 pre-trained word vectors
2. **Filter**: Keep only vectors for our 5000 most common IMDB words
3. **Result**: A focused set of embeddings relevant to our domain

### Why Filter?

- **Relevance**: GloVe contains many rare words not in our corpus
- **Efficiency**: Smaller dataset loads faster in TensorBoard
- **Focus**: Easier to find meaningful patterns when visualizing domain-specific words

### Data Format

The GloVe file format is:
```
word dimension1 dimension2 ... dimension50
the 0.418 0.24968 -0.41242 ... 0.011073
```

We use pandas to parse this efficiently, treating the first column as the index (word) and remaining columns as the 50-dimensional vector.

In [ ]:
df = pd.read_csv(os.path.join('data', 'glove.6B.50d.txt'), header=None, index_col=0, sep=None, error_bad_lines=False, encoding='utf-8')
df.head()

/home/thushv89/anaconda3/envs/manning.tf2/lib/python3.6/site-packages/ipykernel_launcher.py:1: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support sep=None with delim_whitespace=False; you can avoid this warning by specifying engine='python'.
  """Entry point for launching an IPython kernel.
Skipping line 9: field larger than field limit (131072)


,1,2,3,4,5,6,7,8,9,10,...,41,42,43,44,45,46,47,48,49,50
0,,,,,,,,,,,,,,,,,,,,,
the,0.418000,0.249680,-0.41242,0.12170,0.34527,-0.044457,-0.49688,-0.17862,-0.00066,-0.656600,...,-0.298710,-0.157490,-0.347580,-0.045637,-0.44251,0.187850,0.002785,-0.184110,-0.115140,-0.78581
",",0.013441,0.236820,-0.16899,0.40951,0.63812,0.477090,-0.42852,-0.55641,-0.36400,-0.239380,...,-0.080262,0.630030,0.321110,-0.467650,0.22786,0.360340,-0.378180,-0.566570,0.044691,0.30392
.,0.151640,0.301770,-0.16763,0.17684,0.31719,0.339730,-0.43478,-0.31086,-0.44999,-0.294860,...,-0.000064,0.068987,0.087939,-0.102850,-0.13931,0.223140,-0.080803,-0.356520,0.016413,0.10216
of,0.708530,0.570880,-0.47160,0.18048,0.54449,0.726030,0.18157,-0.52393,0.10381,-0.175660,...,-0.347270,0.284830,0.075693,-0.062178,-0.38988,0.229020,-0.216170,-0.225620,-0.093918,-0.80375
to,0.680470,-0.039263,0.30186,-0.17792,0.42962,0.032246,-0.41376,0.13228,-0.29847,-0.085253,...,-0.094375,0.018324,0.210480,-0.030880,-0.19722,0.082279,-0.094340,-0.073297,-0.064699,-0.26044


In [ ]:
print("Full size of Glove: {}".format(df.shape[0]))
df_common = df.loc[df.index.isin(most_common_words)]
print("Size after only considering the most common words: {}".format(df_common.shape))

Full size of Glove: 399694
Size after only considering the most common words: (3595, 50)


## Writing the word vectors in order to be projected on TensorBoard

In [ ]:
# Section 14.5

# Code listing 14.5
from tensorboard.plugins import projector

log_dir=os.path.join('logs', 'embeddings')

# Save the weights we want to analyse as a variable. Note that the first
# value represents any unknown word, which is not in the metadata, so
# we will remove that value.
weights = tf.Variable(df_common.values)
print(weights.shape)
# Create a checkpoint from embedding, the filename and key are
# name of the tensor.
checkpoint = tf.train.Checkpoint(embedding=weights)
checkpoint.save(os.path.join(log_dir, "embedding.ckpt"))

with open(os.path.join(log_dir, 'metadata.tsv'), 'w') as f:
    for w in df_common.index:
        f.write(w+'\n')
        
# Set up config
config = projector.ProjectorConfig()
embedding = config.embeddings.add()
# The name of the tensor will be suffixed by `/.ATTRIBUTES/VARIABLE_VALUE`
#embedding.tensor_name = "embedding/.ATTRIBUTES/VARIABLE_VALUE"
embedding.metadata_path = 'metadata.tsv'
projector.visualize_embeddings(log_dir, config)


(3595, 50)


# Highlighting word vectors

There is a section in the word vectors panel where you can search for specific vectors. You can use regex patterns like the one below to search there and highlight specific vectors.

`(?:fred|larry|mrs\.|mr\.|michelle|sea|denzel|beach|comedy|theater|idiotic|sadistic|marvelous|loving|gorg|bus|truck|lugosi)`

# Separate TensorBoard for word vectors

We also need a separate TensorBoard service (we will use a different port). As visualizing word vectors, the TensorBoard expects to find the data in a very specific folder. Since for the previous TensorBoard we had already defined a different structure, we'll have to view word vectors in a different TensorBoard.

In [ ]:
%tensorboard --logdir logs/embeddings/ --port 6007

2021-05-23 07:37:31.354360: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.11.0
Serving TensorBoard on localhost; to expose to the network, use a proxy or pass --bind_all
TensorBoard 2.4.1 at http://localhost:6006/ (Press CTRL+C to quit)
^C
